# 01 — Exploratory Data Analysis

Run `python -m src.build_dataset` first to generate `data/processed/`.
Visualizes label imbalance, label cardinality, stratification quality, and the top unmapped GoodScents descriptors.

In [ ]:
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import path

proc, inter, figs = path('data_processed'), path('data_interim'), path('reports_figures')
labels = json.loads((proc / 'label_names.json').read_text())
Ytr = sp.load_npz(proc / 'Y_train.npz').toarray()
Yte = sp.load_npz(proc / 'Y_test.npz').toarray()
Y = np.vstack([Ytr, Yte])
summary = json.loads((proc / 'dataset_summary.json').read_text())
summary

## Label frequency (class imbalance)

In [ ]:
freq = pd.Series(Y.sum(0), index=labels).sort_values(ascending=False)
ax = freq.plot(kind='bar', figsize=(16, 4))
ax.set_ylabel('positive count'); ax.set_title('Odor label frequency')
plt.tight_layout(); plt.savefig(figs / 'label_frequency.png', dpi=150); plt.show()
freq.describe()

## Label cardinality (labels per molecule)

In [ ]:
card = Y.sum(1)
plt.figure(figsize=(7, 4))
sns.histplot(card, bins=range(0, int(card.max()) + 2))
plt.xlabel('labels per molecule'); plt.title('Label cardinality')
plt.tight_layout(); plt.savefig(figs / 'label_cardinality.png', dpi=150); plt.show()
print('mean labels/molecule:', round(float(card.mean()), 3))

## Stratification check: train vs test positive rate per label

In [ ]:
rate = pd.DataFrame({'train': Ytr.mean(0), 'test': Yte.mean(0)}, index=labels)
plt.figure(figsize=(5, 5))
plt.scatter(rate['train'], rate['test'], s=10)
lim = float(rate.values.max()) * 1.05
plt.plot([0, lim], [0, lim], 'r--', lw=1)
plt.xlabel('train positive rate'); plt.ylabel('test positive rate')
plt.title('Iterative stratification quality'); plt.tight_layout()
plt.savefig(figs / 'stratification_check.png', dpi=150); plt.show()

## Top unmapped GoodScents descriptors (extend SYNONYMS for these)

In [ ]:
um = pd.read_csv(inter / 'unmapped_descriptors.csv')
um.head(30)